# 03 Retries, Alerts and Monitoring

## Goal

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 Goal</b><br>
Operate jobs reliably: configure <b>retries</b> and <b>timeouts</b>, make retried tasks safe, get <b>notified</b> on failures and slow runs (duration thresholds), <b>repair</b> failed runs without re-running what succeeded, and monitor run history with the UI, the SDK and the <code>system.lakeflow</code> tables.
</div>

## Why it matters

<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Why it matters</b><br>
Every production pipeline fails sometimes: a source is late, a network call times out, a file is malformed. The difference between a minor blip and an incident is whether the job retries safely, whether someone hears about it in time, and how quickly it can be repaired. These operational settings are core to the Lakeflow Jobs exam domain.
</div>

## Concept

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 Reliability settings</b><br>
| Setting | Level | Purpose |<br>|---|---|---|<br>| <code>max_retries</code>, <code>min_retry_interval_millis</code> | Task | Retry transient failures, with a pause between attempts |<br>| <code>retry_on_timeout</code> | Task | Also retry when the task times out |<br>| <code>timeout_seconds</code> | Task or job | Stop runs that hang |<br>| Health rule <code>RUN_DURATION_SECONDS &gt; n</code> | Job | Warn (notify) when a run is slower than expected, without stopping it |<br>| Email / system notifications | Job or task | <code>on_start</code>, <code>on_success</code>, <code>on_failure</code>, <code>on_duration_warning_threshold_exceeded</code>, <code>on_streaming_backlog_exceeded</code> (Slack, Teams, webhooks via notification destinations) |<br>| <code>max_concurrent_runs</code> + queueing | Job | Avoid overlapping runs; queue instead of skipping |<br>| <b>Repair run</b> | Run | Re-run only failed and skipped tasks, optionally with new parameters |<br><br>
<b>Retries are only safe if tasks are idempotent.</b> A retried append writes the data twice.
</div>

## Real-world example

<div style="background:#FFF4DB;border-left:5px solid #E5A100;padding:10px 14px;border-radius:6px;color:#4a3500;">
<b>🏦 Real-world example</b><br>
A job calling a partner API failed about once a week with a 503 error, and someone re-ran the whole 2-hour job each time. They added <code>max_retries = 3</code> with a 2-minute interval on the API task (which upserts with <code>MERGE</code>, so retries are safe), a 30-minute timeout, a duration warning at 90 minutes, and failure notifications to the team channel. The weekly failure disappeared into automatic retries, and when the partner had a real outage, the team was alerted and used <b>repair run</b> to re-run only the 2 failed tasks out of 9.
</div>

## Syntax

```python
jobs.Task(task_key="call_api", notebook_task=...,
          max_retries=3, min_retry_interval_millis=120_000, retry_on_timeout=True, timeout_seconds=1800)

w.jobs.create(..., email_notifications=jobs.JobEmailNotifications(on_failure=["team@example.com"],
                                                                   on_duration_warning_threshold_exceeded=["team@example.com"]),
              health=jobs.JobsHealthRules(rules=[jobs.JobsHealthRule(metric=jobs.JobsHealthMetric.RUN_DURATION_SECONDS,
                                                                    op=jobs.JobsHealthOperator.GREATER_THAN, value=5400)]),
              queue=jobs.QueueSettings(enabled=True), max_concurrent_runs=1)

w.jobs.repair_run(run_id=..., rerun_tasks=["call_api"], job_parameters={...})   # re-run failed parts only
```

## Setup

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Creates the schema, a volume for marker files, and (on Databricks) the SDK client and notebook helper.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
Your user name and folder.
</div>

In [0]:
CATALOG = "workspace"
DB = f"{CATALOG}.workflows"
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {DB}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {DB}.files")
BASE = f"/Volumes/{CATALOG}/workflows/files"

import base64, json, time
from datetime import timedelta
from databricks.sdk import WorkspaceClient
from databricks.sdk.service import jobs, workspace

w = WorkspaceClient()
me = w.current_user.me().user_name
FOLDER = f"/Users/{me}/databricks-learning-workflows"
w.workspace.mkdirs(FOLDER)

def put_notebook(name, source, language=workspace.Language.PYTHON):
    path = f"{FOLDER}/{name}"
    w.workspace.import_(path=path, format=workspace.ImportFormat.SOURCE, language=language,
                        content=base64.b64encode(source.encode()).decode(), overwrite=True)
    return path

def state(r):
    return (r.state.result_state or r.state.life_cycle_state).value

print("user:", me, "| folder:", FOLDER)

## 1. Why retries need idempotent tasks

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Simulates a task that writes its batch and then fails before finishing, so it's retried. Runs it once with a plain <code>append</code> and once with an idempotent <code>MERGE</code>.<br><br>
<b>Why it matters</b><br>A failure can happen <b>after</b> the write succeeded (for example while sending a notification or updating a log). The retry then writes the batch again. With <code>append</code> you get duplicates. With <code>MERGE</code> on a key (or <code>replaceWhere</code>), the retry changes nothing.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
The append target has 6 rows (3 duplicated). The <code>MERGE</code> target has 3.
</div>

In [0]:
from pyspark.sql import functions as F

batch = spark.createDataFrame([(1, 10.0), (2, 20.0), (3, 30.0)], "order_id INT, amount DOUBLE")
for t in ["retry_append", "retry_merge"]:
    spark.sql(f"CREATE OR REPLACE TABLE {DB}.{t} (order_id INT, amount DOUBLE)")

def task_append(attempt):
    batch.write.mode("append").saveAsTable(f"{DB}.retry_append")
    if attempt == 1:
        raise RuntimeError("failed after writing (e.g. while sending a notification)")

def task_merge(attempt):
    batch.createOrReplaceTempView("incoming")
    spark.sql(f"""MERGE INTO {DB}.retry_merge t USING incoming s ON t.order_id = s.order_id
                  WHEN MATCHED THEN UPDATE SET * WHEN NOT MATCHED THEN INSERT *""")
    if attempt == 1:
        raise RuntimeError("failed after writing")

def run_with_retries(task, max_retries=2):
    for attempt in range(1, max_retries + 2):
        try:
            task(attempt)
            return f"succeeded on attempt {attempt}"
        except RuntimeError as e:
            print(f"  attempt {attempt} failed: {e}")
    return "failed"

for name, task in [("append", task_append), ("merge", task_merge)]:
    print(name, "->", run_with_retries(task))
print("rows in retry_append:", spark.table(f"{DB}.retry_append").count())
print("rows in retry_merge: ", spark.table(f"{DB}.retry_merge").count())

## 2. Retries with backoff

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Implements a retry helper with exponential backoff for a flaky call that fails twice before succeeding.<br><br>
<b>Why it matters</b><br>The job-level retry setting re-runs the whole task. Inside a task, transient calls (APIs, external databases) are better retried locally with backoff, so a single flaky call doesn't restart the whole notebook.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
Two failures with growing waits, then success on the third attempt.
</div>

In [0]:
def with_backoff(fn, attempts=4, base_seconds=0.2):
    for i in range(attempts):
        try:
            return fn()
        except ConnectionError as e:
            wait = base_seconds * 2 ** i
            print(f"  call failed ({e}), retrying in {wait:.1f}s")
            time.sleep(wait)
    raise RuntimeError("all attempts failed")

calls = {"n": 0}
def flaky_api():
    calls["n"] += 1
    if calls["n"] < 3:
        raise ConnectionError("503 Service Unavailable")
    return {"status": 200, "attempts": calls["n"]}

print(with_backoff(flaky_api))

## 3. A job with task retries

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Creates a notebook that fails on its first attempt (it leaves a marker file in the volume, and succeeds once the marker exists), and a job whose task retries once after 10 seconds. Runs the job.<br><br>
<b>Why it matters</b><br>You'll see the retry in the run: the task shows two attempts, and the run succeeds. This is what a transient failure looks like when retries are configured.
</div>
<br>
<div style="background:#FFF4DB;border-left:5px solid #E5A100;padding:10px 14px;border-radius:6px;color:#4a3500;">
<b>🆓 Free Edition note</b><br>
Runs on serverless jobs in Free Edition. A retry starts a new attempt, so the run takes a little longer.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
The run ends with <code>SUCCESS</code>. The task list shows the retry: attempt number 1 succeeded (numbering starts at 0, and the failed attempt 0 may be listed too).
</div>

In [0]:
marker = f"{BASE}/flaky_marker_{int(time.time())}"
flaky_path = put_notebook("retry_flaky_task", f'''# Databricks notebook source
import os
marker = "{marker}"
if not os.path.exists(marker):
    open(marker, "w").write("first attempt failed here")
    raise Exception("Simulated transient failure on the first attempt")
dbutils.notebook.exit("succeeded on retry")
''')
job = w.jobs.create(
    name="dbl_03_retries",
    tasks=[jobs.Task(task_key="flaky", notebook_task=jobs.NotebookTask(notebook_path=flaky_path),
                     max_retries=1, min_retry_interval_millis=10_000, timeout_seconds=1800)],
    tags={"course": "databricks-learning", "lesson": "07_03"},
)
run = w.jobs.run_now(job_id=job.job_id).result(timeout=timedelta(minutes=30))
print("run:", state(run), run.run_page_url)
for t in run.tasks:
    print(f"  task {t.task_key}: {state(t)}, attempt number {t.attempt_number}")

## 4. Repair a failed run

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Creates a two-task job where <code>load</code> succeeds and <code>publish</code> fails while the <code>mode</code> parameter is <code>fail</code>. Then <b>repairs</b> the run with <code>mode = ok</code>, re-running only <code>publish</code>.<br><br>
<b>Why it matters</b><br>Repair keeps the successful tasks' results and re-runs only what failed (and what was skipped because of it). On large jobs that saves hours and compute.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
The first run fails at <code>publish</code>. The repair succeeds, and only <code>publish</code> ran again.
</div>

In [0]:
load_path = put_notebook("repair_load", '''# Databricks notebook source
dbutils.notebook.exit("loaded")
''')
publish_path = put_notebook("repair_publish", '''# Databricks notebook source
dbutils.widgets.text("mode", "fail")
if dbutils.widgets.get("mode") == "fail":
    raise Exception("Publishing failed: downstream system unavailable")
dbutils.notebook.exit("published")
''')
rjob = w.jobs.create(
    name="dbl_03_repair",
    parameters=[jobs.JobParameterDefinition(name="mode", default="fail")],
    tasks=[jobs.Task(task_key="load", notebook_task=jobs.NotebookTask(notebook_path=load_path)),
           jobs.Task(task_key="publish", depends_on=[jobs.TaskDependency(task_key="load")],
                     notebook_task=jobs.NotebookTask(notebook_path=publish_path))],
)
first = w.jobs.run_now(job_id=rjob.job_id).result(timeout=timedelta(minutes=30))
print("first run:", state(first), {t.task_key: state(t) for t in first.tasks})

w.jobs.repair_run(run_id=first.run_id, rerun_tasks=["publish"], job_parameters={"mode": "ok"}).result(timeout=timedelta(minutes=30))
repaired = w.jobs.get_run(run_id=first.run_id, include_history=True)
print("after repair:", state(repaired), {t.task_key: state(t) for t in repaired.tasks})
print("repair history entries:", len(repaired.repair_history or []))

## 5. Alerts: notifications and duration warnings

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Adds failure notifications to your email, a duration warning (health rule) of 10 minutes, and queueing with one concurrent run to the retry job, then prints the settings.<br><br>
<b>Why it matters</b><br>Failures should reach a person (or a channel) automatically, and "slower than usual" is often the first sign of trouble (data growth, skew, a stuck source). A duration warning notifies without killing the run, unlike a timeout.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
The job's notification, health and queue settings.
</div>

In [0]:
w.jobs.update(job_id=job.job_id, new_settings=jobs.JobSettings(
    email_notifications=jobs.JobEmailNotifications(on_failure=[me], on_duration_warning_threshold_exceeded=[me]),
    health=jobs.JobsHealthRules(rules=[jobs.JobsHealthRule(metric=jobs.JobsHealthMetric.RUN_DURATION_SECONDS,
                                                           op=jobs.JobsHealthOperator.GREATER_THAN, value=600)]),
    queue=jobs.QueueSettings(enabled=True),
    max_concurrent_runs=1,
))
s = w.jobs.get(job_id=job.job_id).settings
print("on_failure:", s.email_notifications.on_failure)
print("health:", [(r.metric.value, r.op.value, r.value) for r in s.health.rules])
print("queue enabled:", s.queue.enabled, "| max concurrent runs:", s.max_concurrent_runs)

## 6. Monitor run history

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Lists recent runs of the demo jobs with the SDK, then queries the <code>system.lakeflow.job_run_timeline</code> system table for your runs of the last day.<br><br>
<b>Why it matters</b><br>The SDK is handy for one job. System tables answer fleet-wide questions: "which jobs failed most this week?", "which runs are getting slower?".
</div>
<br>
<div style="background:#FFF4DB;border-left:5px solid #E5A100;padding:10px 14px;border-radius:6px;color:#4a3500;">
<b>🆓 Free Edition note</b><br>
<code>system.lakeflow</code> tables may not be available in Free Edition. The cell prints a message if so.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
Recent runs with their results and durations, and (if available) rows from the system table.
</div>

In [0]:
for j in [job, rjob]:
    for r in w.jobs.list_runs(job_id=j.job_id, limit=5):
        secs = ((r.end_time or 0) - (r.start_time or 0)) / 1000
        print(f"job {j.job_id} run {r.run_id}: {state(r):<10} {secs:6.0f}s")

try:
    display(spark.sql("""
        SELECT job_id, run_id, result_state, period_start_time, period_end_time
        FROM system.lakeflow.job_run_timeline
        WHERE period_start_time >= current_timestamp() - INTERVAL 1 DAY
        ORDER BY period_start_time DESC LIMIT 20
    """))
except Exception as e:
    print("system.lakeflow tables not available here:", type(e).__name__)

## 7. Clean up

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Deletes the demo jobs and the marker file.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
<code>jobs deleted</code>.
</div>

In [0]:
for j in [job, rjob]:
    w.jobs.delete(job_id=j.job_id)
dbutils.fs.rm(marker)
print("jobs deleted")

## Under the hood

```
task attempt 0 ── fails ──▶ wait min_retry_interval ──▶ attempt 1 ── succeeds ──▶ downstream tasks
                                                     └─ fails, retries exhausted ──▶ task FAILED ──▶ downstream SKIPPED (ALL_SUCCESS)
                                                                                    └─ on_failure notifications
run duration > health threshold ──▶ on_duration_warning notification (run continues)
run duration > timeout_seconds  ──▶ TIMEDOUT (retried only if retry_on_timeout)
repair run ──▶ same run ID, new repair attempt: failed + skipped tasks re-run, successful ones kept
```

In [0]:
spark.sql(f"DESCRIBE HISTORY {DB}.retry_append").select("version", "operation").show()

## Common mistakes and troubleshooting log

<div style="background:#FDEAEA;border-left:5px solid #D64545;padding:10px 14px;border-radius:6px;color:#4a1414;">
<b>⛔ Problem: retries create duplicate data</b><br>
The task isn't idempotent. Use <code>MERGE</code>, <code>replaceWhere</code> or overwrite of the processed slice.<br><br>
<b>⛔ Problem: retries on a bug</b><br>Retries help transient failures only. A code or data bug fails every attempt and delays the alert. Keep retries low (1–3).<br><br>
<b>⛔ Problem: a hung job runs for days</b><br>Set <code>timeout_seconds</code> on tasks or the job.<br><br>
<b>⛔ Problem: re-running the whole job after one task failed</b><br>Use <b>Repair run</b>.<br><br>
<b>⛔ Problem: alerts go to one person's inbox</b><br>Notify a team channel or a shared address through notification destinations.<br><br>
<b>✍️ My own problems</b><br>(add yours here)
</div>

## Interview Q&A

<div style="background:#F1E9FF;border-left:5px solid #7A4DD8;padding:10px 14px;border-radius:6px;color:#2d1a57;">
<b>🎤 1. How do you handle transient failures in Databricks jobs?</b><br>
Configure task retries with an interval (<code>max_retries</code>, <code>min_retry_interval_millis</code>), and retry individual external calls inside the task with exponential backoff. Most importantly, make the task idempotent (<code>MERGE</code>, <code>replaceWhere</code>) so a retry can't duplicate data.<br><br>

<b>🎤 2. What is a repair run?</b><br>
Re-running only the failed and skipped tasks of a job run, keeping the successful ones, optionally with changed parameters. It saves time and compute compared to re-running the whole job.<br><br>

<b>🎤 3. What's the difference between a timeout and a duration warning?</b><br>
A timeout stops the task or job when it exceeds the limit. A duration warning (a health rule on run duration) sends a notification but lets the run continue.<br><br>

<b>🎤 4. How would you monitor many jobs?</b><br>
Notifications on failure and duration thresholds to team channels, the Jobs UI run history, and queries on <code>system.lakeflow</code> tables (job runs and task runs) for failure rates and duration trends, often in a dashboard with alerts.<br><br>

<b>🎤 5. Why limit concurrent runs and enable queueing?</b><br>
To prevent overlapping runs of the same job from processing the same data concurrently. With queueing, a trigger that arrives while a run is active waits instead of being skipped.
</div>

## Certification corner

<div style="background:#E6F6F7;border-left:5px solid #0E8C8C;padding:10px 14px;border-radius:6px;color:#0b3536;">
<b>🎓 Exam-style questions (Data Engineer Associate)</b><br><br>
<b>Q1. In a 10-task job, 2 tasks failed because of a temporary network issue. What is the most efficient way to complete the run?</b><br>
A. Re-run the entire job<br>
B. Use <b>Repair run</b> to re-run only the failed (and skipped) tasks<br>
C. Clone the job and run the clone<br>
D. Delete the run history and start again<br>
<details><summary><b>Show answer</b></summary><b>B.</b></details><br><br>

<b>Q2. A team wants to be notified when a job runs longer than 2 hours, without stopping it. What should they configure?</b><br>
A. <code>timeout_seconds = 7200</code><br>
B. A run duration health rule with a duration warning notification<br>
C. <code>max_retries = 2</code><br>
D. A file arrival trigger<br>
<details><summary><b>Show answer</b></summary><b>B.</b></details><br><br>

<b>Q3. Which property must a task have for automatic retries to be safe?</b><br>
A. It must run on an all-purpose cluster<br>
B. It must be idempotent<br>
C. It must be written in SQL<br>
D. It must have no parameters<br>
<details><summary><b>Show answer</b></summary><b>B.</b></details>
</div>

## Practice problem

<div style="background:#FFEFE5;border-left:5px solid #F07D2E;padding:10px 14px;border-radius:6px;color:#4a2300;">
<b>🧪 Without copying from above:</b>
<ol><li>Change <code>run_with_retries</code> to stop retrying when the error is a <code>ValueError</code> (a data bug), and retry only <code>ConnectionError</code></li><li>Write an idempotent version of <code>task_append</code> using <code>replaceWhere</code> on <code>order_id</code> ranges, and check the row count after a retry</li><li>(Databricks) Create a job with a task that always fails, <code>max_retries = 2</code> and a 5-second interval, run it, and print the attempt numbers from the run</li></ol>
</div>

In [0]:
# Your solution for the practice problem goes here.
# Add a comment above each step explaining what it does.

## Solution

<div style="background:#FFF4DB;border-left:5px solid #E5A100;padding:10px 14px;border-radius:6px;color:#4a3500;">
<b>⚠️ Try it yourself first</b><br>
Only look at this after you have a working answer of your own.
</div>

In [0]:
# 1. Retry transient errors only
def run_with_smart_retries(task, max_retries=2):
    for attempt in range(1, max_retries + 2):
        try:
            return task(attempt)
        except ConnectionError as e:
            print(f"  transient, retrying: {e}")
        except ValueError as e:
            print(f"  data bug, not retrying: {e}")
            raise
    raise RuntimeError("retries exhausted")

def bad_data(attempt):
    raise ValueError("amount column contains text")

try:
    run_with_smart_retries(bad_data)
except ValueError:
    print("failed fast on a data bug")

# 2. Idempotent with replaceWhere
spark.sql(f"CREATE OR REPLACE TABLE {DB}.retry_replace (order_id INT, amount DOUBLE)")
def task_replace(attempt):
    batch.write.mode("overwrite").option("replaceWhere", "order_id BETWEEN 1 AND 3").saveAsTable(f"{DB}.retry_replace")
    if attempt == 1:
        raise RuntimeError("failed after writing")
print(run_with_retries(task_replace), "| rows:", spark.table(f"{DB}.retry_replace").count())

In [0]:
# 3. Always failing task with retries
fail_path = put_notebook("retry_always_fails", '''# Databricks notebook source
raise Exception("always fails")
''')
fjob = w.jobs.create(name="dbl_03_always_fails", tasks=[jobs.Task(
    task_key="boom", notebook_task=jobs.NotebookTask(notebook_path=fail_path), max_retries=2, min_retry_interval_millis=5_000)])
frun = w.jobs.run_now(job_id=fjob.job_id).result(timeout=timedelta(minutes=30))
print("run:", state(frun))
full = w.jobs.get_run(run_id=frun.run_id)
print("attempts:", sorted(t.attempt_number for t in full.tasks))
w.jobs.delete(job_id=fjob.job_id)

## Summary and cheat sheet

<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Key takeaways</b>
<ul><li>Retries (with an interval) for transient failures, and only with idempotent tasks</li><li>Retry external calls inside tasks with backoff, and fail fast on data bugs</li><li>Timeouts stop hung runs. Duration health rules warn about slow ones</li><li>Notify teams on failure. Queue runs and limit concurrency</li><li>Repair runs re-run only what failed</li><li>Monitor with run history, the SDK and <code>system.lakeflow</code> tables</li></ul>
</div>

| Need | Setting |
|---|---|
| Retry a task | `max_retries`, `min_retry_interval_millis` |
| Stop a hung task | `timeout_seconds` (+ `retry_on_timeout`) |
| Slow-run alert | `health` rule `RUN_DURATION_SECONDS` + `on_duration_warning_threshold_exceeded` |
| Failure alert | `email_notifications.on_failure`, notification destinations |
| No overlaps | `max_concurrent_runs=1`, `queue.enabled=True` |
| Fix a failed run | `w.jobs.repair_run(run_id, rerun_tasks=[...])` |

## Git commit

```
git add 07_workflows/03_retries_alerts_monitoring.ipynb
git commit -m "add 07_03 retries alerts and monitoring notebook"
```